# 01 — Grid Definition

Defines a regular grid over Manhattan and derives the **Y variable** (`zone_type`) from PLUTO `landuse`.

**Method:**
1. Load PLUTO, filter to selected boroughs (default: Manhattan)
2. Compute convex hull of PLUTO lot coordinates as land boundary
3. Generate a 150m x 150m regular grid covering the bounding box
4. Clip grid to convex hull, dropping water/empty cells
5. Assign PLUTO lots to grid cells via grid arithmetic
6. Compute area-weighted `landuse` distribution per cell → derive zone_type

**Output columns:** `cell_id`, `cell_lat`, `cell_lon`, `zone_type`, `cell_lot_count`

**Output file:** `csv/01_grid_definition.csv`

In [ ]:
# ── Papermill parameters ──────────────────────────────
GRID_CONFIG = "grid.json"

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import math
from scipy.spatial import ConvexHull
from shapely.geometry import Polygon, Point

with open(GRID_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

BOROUGH_FILTER = config["borough_filter"]
BOROUGH_CODES = config["borough_codes"]
PLUTO_PATH = config["pluto_path"]
CELL_SIZE_M = config["grid_cell_size_m"]
MIN_LOTS = config["min_lots_per_cell"]
CSV_DIR = config.get("csv_dir", "csv")

os.makedirs(CSV_DIR, exist_ok=True)

boro_code_filter = [str(BOROUGH_CODES[b]) for b in BOROUGH_FILTER]
print(f"Boroughs: {BOROUGH_FILTER} (codes: {boro_code_filter})")
print(f"Grid cell size: {CELL_SIZE_M}m x {CELL_SIZE_M}m")
print(f"Min lots per cell: {MIN_LOTS}")
print(f"CSV output dir: {CSV_DIR}/")

In [ ]:
# ── Load PLUTO (only needed columns) ──────────────────
PLUTO_COLS = [
    "borocode", "landuse", "lotarea",
    "latitude", "longitude", "borough",
]

df_pluto = pd.read_csv(PLUTO_PATH, usecols=PLUTO_COLS, dtype={"landuse": str})
df_pluto = df_pluto[df_pluto["borocode"].astype(str).isin(boro_code_filter)].copy()

df_pluto["lotarea"] = pd.to_numeric(df_pluto["lotarea"], errors="coerce").fillna(0)
df_pluto["latitude"] = pd.to_numeric(df_pluto["latitude"], errors="coerce")
df_pluto["longitude"] = pd.to_numeric(df_pluto["longitude"], errors="coerce")
df_pluto = df_pluto.dropna(subset=["latitude", "longitude"]).copy()

print(f"PLUTO lots after filter: {len(df_pluto):,}")
print(f"Lat range: {df_pluto['latitude'].min():.4f} - {df_pluto['latitude'].max():.4f}")
print(f"Lon range: {df_pluto['longitude'].min():.4f} - {df_pluto['longitude'].max():.4f}")

In [ ]:
# ── Map PLUTO landuse codes to zone type categories ───
#
# PLUTO landuse codes:
#   01 = One & Two Family Buildings
#   02 = Multi-Family Walk-Up Buildings
#   03 = Multi-Family Elevator Buildings
#   04 = Mixed Residential & Commercial
#   05 = Commercial & Office Buildings
#   06 = Industrial & Manufacturing
#   07 = Transportation & Utility
#   08 = Public Facilities & Institutions
#   09 = Open Space & Outdoor Recreation
#   10 = Parking Facilities
#   11 = Vacant Land

LANDUSE_TO_ZONE = {
    "01": "Residential",
    "02": "Residential",
    "03": "Residential",
    "04": "Mixed-Use",
    "05": "Commercial",
    "06": "Industrial",
    "07": "Infrastructure",
    "08": "Institutional",
    "09": "Open Space",
    "10": "Infrastructure",
    "11": "Infrastructure",
}

PLURALITY_THRESHOLD = 0.40

df_pluto["zone_cat"] = df_pluto["landuse"].map(LANDUSE_TO_ZONE).fillna("Unknown")
print("Landuse mapping applied.")
print(df_pluto["zone_cat"].value_counts().to_string())

In [ ]:
# ── Generate regular grid ─────────────────────────────

# Convert cell size from meters to degrees at Manhattan's latitude
REF_LAT = df_pluto["latitude"].mean()
LAT_STEP = CELL_SIZE_M / 111_000  # 1 degree lat ≈ 111 km
LON_STEP = CELL_SIZE_M / (111_000 * math.cos(math.radians(REF_LAT)))

print(f"Reference latitude: {REF_LAT:.4f}")
print(f"Grid steps: lat={LAT_STEP:.6f} deg, lon={LON_STEP:.6f} deg")

# Bounding box with small buffer
BUFFER = LAT_STEP  # one cell buffer
LAT_MIN = df_pluto["latitude"].min() - BUFFER
LAT_MAX = df_pluto["latitude"].max() + BUFFER
LON_MIN = df_pluto["longitude"].min() - BUFFER
LON_MAX = df_pluto["longitude"].max() + BUFFER

# Grid row/col ranges
n_rows = int(math.ceil((LAT_MAX - LAT_MIN) / LAT_STEP))
n_cols = int(math.ceil((LON_MAX - LON_MIN) / LON_STEP))
print(f"Grid dimensions: {n_rows} rows x {n_cols} cols = {n_rows * n_cols:,} total cells")

# Build convex hull from PLUTO lot coordinates for land clipping
coords = df_pluto[["longitude", "latitude"]].values
hull = ConvexHull(coords)
hull_polygon = Polygon(coords[hull.vertices])
print(f"Convex hull area: {hull_polygon.area:.6f} sq degrees")

In [ ]:
# ── Clip grid to land + assign PLUTO lots to cells ────

# Assign each PLUTO lot to a grid cell
df_pluto["grid_row"] = ((df_pluto["latitude"] - LAT_MIN) / LAT_STEP).astype(int)
df_pluto["grid_col"] = ((df_pluto["longitude"] - LON_MIN) / LON_STEP).astype(int)
df_pluto["cell_id"] = "r" + df_pluto["grid_row"].astype(str).str.zfill(4) + "_c" + df_pluto["grid_col"].astype(str).str.zfill(4)

# Find cells that have PLUTO lots assigned
cell_groups = df_pluto.groupby(["grid_row", "grid_col"])
print(f"Cells with at least 1 lot: {cell_groups.ngroups:,}")

# Build cell records — only keep cells with enough lots AND inside convex hull
cell_records = []
skipped_hull = 0
skipped_lots = 0

for (row, col), group in cell_groups:
    lot_count = len(group)
    
    # Cell center coordinates
    cell_lat = LAT_MIN + (row + 0.5) * LAT_STEP
    cell_lon = LON_MIN + (col + 0.5) * LON_STEP
    
    # Check if cell center is inside convex hull
    if not hull_polygon.contains(Point(cell_lon, cell_lat)):
        skipped_hull += 1
        continue
    
    # Minimum lot threshold
    if lot_count < MIN_LOTS:
        skipped_lots += 1
        continue
    
    cell_id = f"r{row:04d}_c{col:04d}"
    total_area = group["lotarea"].sum()
    
    # Zone type by area-weighted plurality
    if total_area == 0:
        zone_type = "Mixed-Use"
    else:
        zone_area = group.groupby("zone_cat")["lotarea"].sum()
        dominant = zone_area.idxmax()
        dominant_ratio = zone_area[dominant] / total_area
        
        if dominant in ("Infrastructure", "Unknown"):
            remaining = zone_area.drop(["Infrastructure", "Unknown"], errors="ignore")
            if len(remaining) > 0:
                dominant = remaining.idxmax()
                dominant_ratio = remaining[dominant] / total_area
            else:
                zone_type = "Mixed-Use"
                dominant_ratio = 0
        
        if dominant_ratio >= PLURALITY_THRESHOLD:
            zone_type = dominant
        else:
            zone_type = "Mixed-Use"
    
    cell_records.append({
        "cell_id": cell_id,
        "cell_lat": round(cell_lat, 7),
        "cell_lon": round(cell_lon, 7),
        "zone_type": zone_type,
        "cell_lot_count": lot_count,
    })

df_grid = pd.DataFrame(cell_records)
print(f"\nSkipped (outside hull): {skipped_hull}")
print(f"Skipped (< {MIN_LOTS} lots): {skipped_lots}")
print(f"Final grid cells: {len(df_grid):,}")

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/01_grid_definition.csv"
df_grid.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_grid)} rows x {df_grid.shape[1]} cols)")
df_grid.head(10)

In [ ]:
# ── Summary statistics ────────────────────────────────
print("Zone type breakdown:")
for zt in sorted(df_grid["zone_type"].unique()):
    subset = df_grid[df_grid["zone_type"] == zt]
    print(f"  {zt:<20s} {len(subset):>5d} cells  "
          f"(avg {subset['cell_lot_count'].mean():.0f} lots/cell)")

print(f"\nTotal cells: {len(df_grid):,}")
print(f"Lat range: {df_grid['cell_lat'].min():.4f} - {df_grid['cell_lat'].max():.4f}")
print(f"Lon range: {df_grid['cell_lon'].min():.4f} - {df_grid['cell_lon'].max():.4f}")
print(f"\nCell area: {(CELL_SIZE_M/1000)**2:.4f} km2")
print(f"Grid step: {LAT_STEP:.6f} lat, {LON_STEP:.6f} lon")